# Feature Quality and Data Preparation Audit

## Objective

This notebook examines the quality, construction and suitability of the predictor variables used in the traditional bank usage prediction model.

The audit focuses on:

1. Education variable coding and missingness.
2. Income variables and derived income measures.
3. Wealth variable construction.
4. Financial literacy measurement.
5. Physical access and nearest-provider variables.

The objective is to identify potential measurement errors, inappropriate transformations, missing-data issues and possible sources of gender-related prediction disparities before modifying the modelling pipeline.

The original processed dataset and cleaning script will remain unchanged during this investigation.

In [3]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(r"C:\Users\maken\Documents\Strathmore Graduate Studies\DSA 8401 - Applied Machine Learning in Data Science\Assignments\Capstone Project\finaccess-fairness\data\processed\finaccess_selected.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully
Shape: (20871, 48)
Columns: ['respondent_id', 'county', 'cluster_id', 'location_type', 'hh_head_gender', 'age', 'relationship_to_hh_head', 'gender', 'household_size', 'num_children', 'num_children_under5', 'decides_daily_spending', 'decides_major_spending', 'skipped_meals_last_month', 'can_read_sms', 'income_from_farming', 'income_from_regular_job', 'income_from_casual_work', 'income_from_own_business', 'relies_on_family_support', 'income_last_month', 'income_highest_month', 'income_lowest_month', 'phone_access_type', 'phone_internet_capable', 'internet_use_frequency', 'network_quality', 'nearest_provider_type', 'cost_to_nearest_provider', 'distance_to_nearest_provider', 'num_indigenous_cattle', 'owns_house', 'owns_land', 'land_ownership_document', 'survey_weight', 'livelihood_category', 'wealth_quintile', 'education_level', 'marital_status', 'has_any_id', 'owns_phone', 'mobile_money_access', 'num_income_sources', 'target_formal_access', 'target_bank_account

In [4]:

print("Number of observations:", df.shape[0])
print("Number of variables:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five observations:")
display(df.head())

Number of observations: 20871
Number of variables: 48

Column names:
['respondent_id', 'county', 'cluster_id', 'location_type', 'hh_head_gender', 'age', 'relationship_to_hh_head', 'gender', 'household_size', 'num_children', 'num_children_under5', 'decides_daily_spending', 'decides_major_spending', 'skipped_meals_last_month', 'can_read_sms', 'income_from_farming', 'income_from_regular_job', 'income_from_casual_work', 'income_from_own_business', 'relies_on_family_support', 'income_last_month', 'income_highest_month', 'income_lowest_month', 'phone_access_type', 'phone_internet_capable', 'internet_use_frequency', 'network_quality', 'nearest_provider_type', 'cost_to_nearest_provider', 'distance_to_nearest_provider', 'num_indigenous_cattle', 'owns_house', 'owns_land', 'land_ownership_document', 'survey_weight', 'livelihood_category', 'wealth_quintile', 'education_level', 'marital_status', 'has_any_id', 'owns_phone', 'mobile_money_access', 'num_income_sources', 'target_formal_access', 'target

,respondent_id,county,cluster_id,location_type,hh_head_gender,age,relationship_to_hh_head,gender,household_size,num_children,...,marital_status,has_any_id,owns_phone,mobile_money_access,num_income_sources,target_formal_access,target_bank_account,experienced_shock,financial_literacy_score,disability_status
0,21-61-43-44,Garissa,225,Urban,Male,29,Spouse,Female,5,1,...,Married/Living with partner,"Has ID (National Identity Card (ID),Passport,A...",Yes,Yes,1,Yes,Usage,No,All correct,Without Disability
1,87-78-54-60,Garissa,223,Urban,Male,60,Other relative,Female,11,8,...,Married/Living with partner,"Has ID (National Identity Card (ID),Passport,A...",Yes,Yes,1,Yes,Non-usage,No,Two correct,Without Disability
2,32-71-19-84,Busia,1514,Urban,Female,35,Head of household,Female,2,5,...,Divorced/separated,"Has ID (National Identity Card (ID),Passport,A...",Yes,Yes,1,Yes,Non-usage,Yes,All correct,Without Disability
3,48-18-78-99,Kiambu,796,Urban,Male,24,Head of household,Male,1,NaN,...,Single/Never Married,"Has ID (National Identity Card (ID),Passport,A...",Yes,Yes,1,Yes,Non-usage,No,All correct,Without Disability
4,50-82-98-89,Murang'a,763,Urban,Female,20,Head of household,Female,1,NaN,...,Single/Never Married,"Has ID (National Identity Card (ID),Passport,A...",Yes,Yes,1,Yes,Usage,No,All correct,Without Disability


In [5]:
# Dataset information
print("DATASET INFORMATION")
print("-" * 50)

df.info()

# Data types
print("\nDATA TYPES")
display(df.dtypes.to_frame(name="Data Type"))

# Duplicate observations
print("\nDUPLICATE OBSERVATIONS")
print("Number of duplicate rows:", df.duplicated().sum())

# Unique values in each column
print("\nNUMBER OF UNIQUE VALUES")
display(
    df.nunique(dropna=False)
      .sort_values()
      .to_frame(name="Unique Values")
)

# Basic descriptive statistics
print("\nNUMERICAL DESCRIPTIVE STATISTICS")
display(df.describe().T)

DATASET INFORMATION
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 20871 entries, 0 to 20870
Data columns (total 48 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   respondent_id                 20871 non-null  str  
 1   county                        20871 non-null  str  
 2   cluster_id                    20871 non-null  int64
 3   location_type                 20871 non-null  str  
 4   hh_head_gender                20871 non-null  str  
 5   age                           20871 non-null  int64
 6   relationship_to_hh_head       20871 non-null  str  
 7   gender                        20871 non-null  str  
 8   household_size                20871 non-null  int64
 9   num_children                  16068 non-null  str  
 10  num_children_under5           11647 non-null  str  
 11  decides_daily_spending        20871 non-null  str  
 12  decides_major_spending      

,Data Type
respondent_id,str
county,str
cluster_id,int64
location_type,str
hh_head_gender,str
age,int64
relationship_to_hh_head,str
gender,str
household_size,int64
num_children,str



DUPLICATE OBSERVATIONS
Number of duplicate rows: 0

NUMBER OF UNIQUE VALUES


,Unique Values
location_type,2
hh_head_gender,2
gender,2
income_from_farming,2
relies_on_family_support,2
income_from_own_business,2
income_from_casual_work,2
income_from_regular_job,2
disability_status,2
mobile_money_access,2



NUMERICAL DESCRIPTIVE STATISTICS


,count,mean,std,min,25%,50%,75%,max
cluster_id,20871.0,927.665613,534.388410,1.0,451.0,934.0,1393.0,1849.0
age,20871.0,39.271429,17.210021,16.0,26.0,35.0,50.0,105.0
household_size,20871.0,4.215946,2.512658,1.0,2.0,4.0,6.0,20.0
income_from_farming,20871.0,0.301040,0.458721,0.0,0.0,0.0,1.0,1.0
income_from_regular_job,20871.0,0.120933,0.326058,0.0,0.0,0.0,0.0,1.0
income_from_casual_work,20871.0,0.308035,0.461692,0.0,0.0,0.0,1.0,1.0
income_from_own_business,20871.0,0.189593,0.391989,0.0,0.0,0.0,0.0,1.0
relies_on_family_support,20871.0,0.324805,0.468313,0.0,0.0,0.0,1.0,1.0
survey_weight,20871.0,1463.902880,1740.019694,20.0,553.0,994.0,1773.0,73513.0
num_income_sources,20871.0,1.346078,0.602394,1.0,1.0,1.0,2.0,7.0


In [6]:
print("TARGET VARIABLE AUDIT")
print("-" * 50)

target = "target_bank_account"

if target in df.columns:

    print("\nTarget value counts:")
    display(df[target].value_counts(dropna=False))

    print("\nTarget proportions:")
    display(
        df[target]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

else:
    print(f"Target variable '{target}' was not found.")


# Identify the gender variable
gender_candidates = ["gender", "sex", "A13", "Gender", "Sex"]

gender_col = next(
    (col for col in gender_candidates if col in df.columns),
    None
)

print("\nGender variable identified:", gender_col)

if gender_col is not None:

    print("\nGender distribution:")
    display(df[gender_col].value_counts(dropna=False))

    print("\nGender distribution in percentages:")
    display(
        df[gender_col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    print("\nTarget distribution by gender:")
    display(
        pd.crosstab(
            df[gender_col],
            df[target],
            normalize="index"
        ).mul(100).round(2)
    )

else:
    print("No recognised gender column was found.")

TARGET VARIABLE AUDIT
--------------------------------------------------

Target value counts:


target_bank_account
Non-usage    15672
Usage         5199
Name: count, dtype: int64


Target proportions:


target_bank_account
Non-usage    75.09
Usage        24.91
Name: proportion, dtype: float64


Gender variable identified: gender

Gender distribution:


gender
Female    12326
Male       8545
Name: count, dtype: int64


Gender distribution in percentages:


gender
Female    59.06
Male      40.94
Name: proportion, dtype: float64


Target distribution by gender:


target_bank_account,Non-usage,Usage
gender,,
Female,79.50,20.50
Male,68.73,31.27


In [7]:
print("MISSING VALUE AUDIT")
print("-" * 50)

# Overall missing values
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

missing = missing[missing["Missing Count"] > 0]

missing = missing.sort_values(
    "Missing Percentage",
    ascending=False
)

display(missing.round(2))


# Missingness by gender
if gender_col is not None:

    print("\nMISSINGNESS BY GENDER")
    print("-" * 50)

    missing_by_gender = []

    for col in df.columns:

        for gender in df[gender_col].dropna().unique():

            subset = df[df[gender_col] == gender]

            missing_by_gender.append({
                "Variable": col,
                "Gender": gender,
                "Missing Count": subset[col].isnull().sum(),
                "Missing Percentage":
                    subset[col].isnull().mean() * 100
            })

    missing_gender_df = pd.DataFrame(missing_by_gender)

    display(
        missing_gender_df[
            missing_gender_df["Missing Count"] > 0
        ].round(2)
        .sort_values(
            ["Variable", "Gender"]
        )
    )

MISSING VALUE AUDIT
--------------------------------------------------


,Missing Count,Missing Percentage
num_children_under5,9224,44.20
num_children,4803,23.01
education_level,3078,14.75



MISSINGNESS BY GENDER
--------------------------------------------------


,Variable,Gender,Missing Count,Missing Percentage
74,education_level,Female,2246,18.22
75,education_level,Male,832,9.74
18,num_children,Female,1993,16.17
19,num_children,Male,2810,32.88
20,num_children_under5,Female,5995,48.64
21,num_children_under5,Male,3229,37.79


In [8]:
education_col = "education_level"

print("EDUCATION VARIABLE AUDIT")
print("-" * 50)

if education_col in df.columns:

    print("\nEducation categories:")
    display(df[education_col].value_counts(dropna=False))

    print("\nEducation category percentages:")
    display(
        df[education_col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    print("\nEducation distribution by gender:")
    display(
        pd.crosstab(
            df[gender_col],
            df[education_col],
            normalize="index"
        ).mul(100).round(2)
    )

    print("\nMissing education values by gender:")
    display(
        df.groupby(gender_col, dropna=False)[education_col]
        .apply(lambda x: x.isnull().sum())
        .to_frame(name="Missing Count")
    )

else:
    print("education_level was not found.")

EDUCATION VARIABLE AUDIT
--------------------------------------------------

Education categories:


education_level
Primary      7684
Secondary    7345
NaN          3078
Tertiary     2755
Other           9
Name: count, dtype: int64


Education category percentages:


education_level
Primary      36.82
Secondary    35.19
NaN          14.75
Tertiary     13.20
Other         0.04
Name: proportion, dtype: float64


Education distribution by gender:


education_level,Other,Primary,Secondary,Tertiary
gender,,,,
Female,0.04,45.14,40.52,14.31
Male,0.06,40.63,42.28,17.02



Missing education values by gender:


,Missing Count
gender,
Female,2246
Male,832


In [9]:
if education_col in df.columns:

    print("EDUCATION CATEGORY REVIEW")
    print("-" * 50)

    # Examine all distinct values
    print("Distinct education values:")
    print(df[education_col].dropna().unique())

    # Check for possible unexpected categories
    expected_categories = [
        "None",
        "Primary",
        "Secondary",
        "Tertiary",
        "Other"
    ]

    observed_categories = df[education_col].dropna().unique()

    unexpected = [
        value for value in observed_categories
        if value not in expected_categories
    ]

    print("\nCategories outside the expected list:")
    print(unexpected)

    # Compare education and target
    if target in df.columns:

        print("\nBank usage by education level:")
        display(
            pd.crosstab(
                df[education_col],
                df[target],
                normalize="index"
            ).mul(100).round(2)
        )

EDUCATION CATEGORY REVIEW
--------------------------------------------------
Distinct education values:
<ArrowStringArray>
['Tertiary', 'Primary', 'Secondary', 'Other']
Length: 4, dtype: str

Categories outside the expected list:
[]

Bank usage by education level:


target_bank_account,Non-usage,Usage
education_level,,
Other,88.89,11.11
Primary,84.32,15.68
Secondary,75.70,24.30
Tertiary,34.12,65.88


In [13]:
income_cols = [
    "income_last_month",
    "income_highest_month",
    "income_lowest_month",
    "income_volatility",
    "num_income_sources"
]

available_income_cols = [
    col for col in income_cols
    if col in df.columns
]

print("INCOME VARIABLE AUDIT")
print("-" * 50)

print("Income variables found:")
print(available_income_cols)

print("\nDescriptive statistics:")
display(df[available_income_cols].describe().T)

print("\nMissing values:")
display(
    df[available_income_cols]
    .isnull()
    .sum()
    .to_frame(name="Missing Count")
)

print("\nIncome variables by gender:")

if gender_col is not None:

    income_by_gender = df[[gender_col] + available_income_cols].copy()

    for col in available_income_cols:
        income_by_gender[col] = pd.to_numeric(
            income_by_gender[col],
            errors="coerce"
        )

    display(
        income_by_gender.groupby(gender_col)[available_income_cols]
        .agg(["mean", "median", "min", "max"])
        .round(2)
    )

INCOME VARIABLE AUDIT
--------------------------------------------------
Income variables found:
['income_last_month', 'income_highest_month', 'income_lowest_month', 'num_income_sources']

Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
num_income_sources,20871.0,1.346078,0.602394,1.0,1.0,1.0,2.0,7.0



Missing values:


,Missing Count
income_last_month,0
income_highest_month,0
income_lowest_month,0
num_income_sources,0



Income variables by gender:


income_last_month                          income_highest_month  \
                    mean  median    min       max                 mean   
gender                                                                   
Female           8273.07  4000.0  100.0  200000.0             13215.33   
Male            13120.73  7000.0  100.0  200000.0             21125.97   

                                 income_lowest_month                          \
         median    min       max                mean  median   min       max   
gender                                                                         
Female   6000.0  200.0  300000.0             4726.02  2000.0  50.0  100000.0   
Male    10000.0  200.0  300000.0             7673.01  4000.0  50.0  100000.0   

       num_income_sources                 
                     mean median min max  
gender                                    
Female               1.34    1.0   1   5  
Male                 1.36    1.0   1   7

In [11]:
print("INCOME CONSISTENCY CHECK")
print("-" * 50)

required_income = [
    "income_highest_month",
    "income_lowest_month"
]

if all(col in df.columns for col in required_income):

    highest = df["income_highest_month"]
    lowest = df["income_lowest_month"]

    # Observations where lowest income exceeds highest income
    inconsistent = df[
        (lowest > highest) &
        highest.notna() &
        lowest.notna()
    ]

    print(
        "Observations where lowest income exceeds highest income:",
        len(inconsistent)
    )

    print(
        "Percentage of dataset:",
        round(len(inconsistent) / len(df) * 100, 2), "%"
    )

    print("\nExamples of inconsistent observations:")

    display(
        inconsistent[
            [gender_col, "income_highest_month",
             "income_lowest_month"]
        ].head(20)
    )

    print("\nIncome values equal to zero:")

    display(
        df[required_income]
        .eq(0)
        .sum()
        .to_frame(name="Zero Count")
    )

else:
    print("Required income variables were not found.")

INCOME CONSISTENCY CHECK
--------------------------------------------------
Observations where lowest income exceeds highest income: 6883
Percentage of dataset: 32.98 %

Examples of inconsistent observations:


,gender,income_highest_month,income_lowest_month
1,Female,13000,9500
3,Male,19000,7000
4,Female,25000,3000
6,Female,2000,500
7,Male,30000,5000
8,Female,2000,500
14,Male,15000,5000
17,Male,27000,700
18,Female,2000,500
22,Female,12000,3000



Income values equal to zero:


,Zero Count
income_highest_month,0
income_lowest_month,0


In [15]:
print("INCOME VOLATILITY AUDIT")
print("-" * 50)

import numpy as np
required_income = [
    "income_highest_month",
    "income_lowest_month"
]

if all(col in df.columns for col in required_income):

    highest = df["income_highest_month"]
    lowest = df["income_lowest_month"]

    # Recalculate the ratio for comparison only.
    # This does not overwrite the original variable.

    highest_num = pd.to_numeric(highest, errors="coerce")
    lowest_num = pd.to_numeric(lowest, errors="coerce")

    calculated_ratio = 1 - (lowest_num / highest_num)

    calculated_ratio = calculated_ratio.replace(
        [np.inf, -np.inf],
        np.nan
    )

    print("Original income_volatility statistics:")

    if "income_volatility" in df.columns:
        display(df["income_volatility"].describe())

    print("\nRecalculated ratio statistics:")
    display(calculated_ratio.describe())

    print("\nRecalculated ratio outside the 0-1 range:")

    outside_range = calculated_ratio[
        (calculated_ratio < 0) |
        (calculated_ratio > 1)
    ]

    print("Number of observations:", outside_range.notna().sum())

    print("\nExamples:")
    display(outside_range.head(20))

    print("\nComparison between original and recalculated measures:")

    comparison = pd.DataFrame({
        "Original": df["income_volatility"]
        if "income_volatility" in df.columns
        else np.nan,

        "Recalculated": calculated_ratio
    })

    display(comparison.describe().T)

    print("\nIncome ratio by gender:")

    if gender_col is not None:
        comparison[gender_col] = df[gender_col]

        display(
            comparison.groupby(gender_col)["Recalculated"]
            .describe()
            .round(3)
        )

INCOME VOLATILITY AUDIT
--------------------------------------------------
Original income_volatility statistics:

Recalculated ratio statistics:


count    18160.000000
mean         0.601366
std          0.436893
min        -25.250000
25%          0.500000
50%          0.666667
75%          0.800000
max          0.999286
dtype: float64


Recalculated ratio outside the 0-1 range:
Number of observations: 84

Examples:


352     -0.500000
444     -3.444444
760     -5.400000
1134    -1.000000
1366    -0.012658
1475   -13.814815
1478    -4.500000
1641    -0.200000
1683   -14.000000
1731    -0.777778
1745    -1.500000
1813    -2.500000
1985    -8.090909
2495    -1.000000
2958    -0.500000
4156    -1.000000
4464    -1.000000
4622    -0.500000
4973    -0.166667
5451    -7.333333
dtype: float64


Comparison between original and recalculated measures:


,count,mean,std,min,25%,50%,75%,max
Original,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Recalculated,18160.0,0.601366,0.436893,-25.25,0.5,0.666667,0.8,0.999286



Income ratio by gender:


,count,mean,std,min,25%,50%,75%,max
gender,,,,,,,,
Female,10574.0,0.612,0.302,-9.00,0.500,0.667,0.8,0.999
Male,7586.0,0.586,0.574,-25.25,0.464,0.667,0.8,0.999


In [16]:
wealth_col = "wealth_quintile"

print("WEALTH QUINTILE AUDIT")
print("-" * 50)

if wealth_col in df.columns:

    print("\nWealth quintile categories:")
    display(df[wealth_col].value_counts(dropna=False))

    print("\nWealth quintile percentages:")
    display(
        df[wealth_col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    print("\nWealth quintile by gender:")
    display(
        pd.crosstab(
            df[gender_col],
            df[wealth_col],
            normalize="index"
        ).mul(100).round(2)
    )

    print("\nBank usage by wealth quintile:")
    display(
        pd.crosstab(
            df[wealth_col],
            df[target],
            normalize="index"
        ).mul(100).round(2)
    )

else:
    print("wealth_quintile was not found.")

WEALTH QUINTILE AUDIT
--------------------------------------------------

Wealth quintile categories:


wealth_quintile
Lowest     4973
Second     4478
Middle     4293
Fourth     4232
Highest    2859
#NULL!       36
Name: count, dtype: int64


Wealth quintile percentages:


wealth_quintile
Lowest     23.83
Second     21.46
Middle     20.57
Fourth     20.28
Highest    13.70
#NULL!      0.17
Name: proportion, dtype: float64


Wealth quintile by gender:


wealth_quintile,#NULL!,Fourth,Highest,Lowest,Middle,Second
gender,,,,,,
Female,0.14,20.24,13.52,24.85,20.37,20.87
Male,0.22,20.33,13.95,22.35,20.85,22.29



Bank usage by wealth quintile:


target_bank_account,Non-usage,Usage
wealth_quintile,,
#NULL!,72.22,27.78
Fourth,63.35,36.65
Highest,45.44,54.56
Lowest,92.38,7.62
Middle,76.89,23.11
Second,84.21,15.79


In [19]:
literacy_candidates = [
    "financial_literacy_score",
    "fl_score"
]

literacy_col = next(
    (col for col in literacy_candidates if col in df.columns),
    None
)

print("FINANCIAL LITERACY AUDIT")
print("-" * 50)

print("Financial literacy variable identified:", literacy_col)

if literacy_col is not None:

    print("\nDescriptive statistics:")
    display(df[literacy_col].describe())

    print("\nUnique values:")
    print(df[literacy_col].nunique())

    print("\nFirst 20 distinct values:")
    print(df[literacy_col].dropna().unique()[:20])

    print("\nMissing values:")
    print(df[literacy_col].isnull().sum())

    print("\nDistribution by gender:")

    display(
        df.groupby(gender_col, dropna=False)[literacy_col]
        .value_counts()
        .unstack(fill_value=0)
        .sort_index(axis=1)
    )

    print("\nBank usage by financial literacy score:")

    literacy_score_map = {
        "None correct": 0,
        "One correct": 1,
        "Two correct": 2,
        "All correct": 3
    }

    literacy_scores = df[literacy_col].map(literacy_score_map)

    if literacy_scores.notna().all():
        display(
            df.assign(literacy_score=literacy_scores)
            .groupby(target)["literacy_score"]
            .agg(["count", "mean", "median", "std"])
            .round(3)
        )
    else:
        display(
            pd.crosstab(
                df[target],
                df[literacy_col],
                dropna=False
            ).sort_index(axis=1)
        )

else:
    print("No recognised financial literacy variable was found.")

FINANCIAL LITERACY AUDIT
--------------------------------------------------
Financial literacy variable identified: financial_literacy_score

Descriptive statistics:


count           20871
unique              4
top       Two correct
freq             8580
Name: financial_literacy_score, dtype: object


Unique values:
4

First 20 distinct values:
<ArrowStringArray>
['All correct', 'Two correct', 'One correct', 'None correct']
Length: 4, dtype: str

Missing values:
0

Distribution by gender:


financial_literacy_score,All correct,None correct,One correct,Two correct
gender,,,,
Female,4414,452,2273,5187
Male,3671,261,1220,3393



Bank usage by financial literacy score:


,count,mean,median,std
target_bank_account,,,,
Non-usage,15672,2.095,2.0,0.828
Usage,5199,2.324,2.0,0.758


In [22]:
provider_candidates = [
    "nearest_provider_type",
    "distance_to_nearest_provider",
    "distance_nearest_provider",
    "nearest_provider_distance",
    "cost_to_nearest_provider",
    "cost_nearest_provider",
    "nearest_provider_cost"
]

available_provider_cols = [
    col for col in provider_candidates
    if col in df.columns
]

print("PROVIDER ACCESS AUDIT")
print("-" * 50)

print("Provider variables found:")
print(available_provider_cols)

for col in available_provider_cols:

    print("\n" + "=" * 50)
    print("VARIABLE:", col)

    print("\nData type:", df[col].dtype)

    print("\nNumber of unique values:", df[col].nunique())

    print("\nMissing values:", df[col].isnull().sum())

    print("\nFirst 20 values:")

    if df[col].dtype == "object":
        display(df[col].value_counts(dropna=False).head(20))

    else:
        display(df[col].describe())

    print("\nDistribution by gender:")

    if gender_col is not None:

        if df[col].dtype == "object":
            display(
                pd.crosstab(
                    df[gender_col],
                    df[col],
                    normalize="index"
                ).mul(100).round(2)
            )

        else:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_values = pd.to_numeric(df[col], errors="coerce")
                display(
                    df.assign(_numeric=numeric_values)
                    .groupby(gender_col)["_numeric"]
                    .agg(["count", "mean", "median", "min", "max"])
                    .round(2)
                )
            else:
                display(
                    pd.crosstab(
                        df[gender_col],
                        df[col],
                        dropna=False
                    )
                )

PROVIDER ACCESS AUDIT
--------------------------------------------------
Provider variables found:
['nearest_provider_type', 'distance_to_nearest_provider', 'cost_to_nearest_provider']

VARIABLE: nearest_provider_type

Data type: str

Number of unique values: 9

Missing values: 0

First 20 values:


count                                                 20871
unique                                                    9
top       Mobile Money Agent (for depositing or withdraw...
freq                                                  19859
Name: nearest_provider_type, dtype: object


Distribution by gender:


nearest_provider_type,#NULL!,Bank Agent/ Post Bank Agent/wakala/ajenti wa benki/ wakala/ajenti waPostbank,Commercial Bank/Microfinance Bank/ Post Bank Branch/ Head Office/ ATM/ Benki/taw,"Don't know/Sijui.(DO NOT READ OUT)""","Insurance company/ agent/""Kampuni la bima/wakalaau ajenti""","Microfinance Institution/""Taasisi ndogo la kifedha""",Mobile Money Agent (for depositing or withdrawing cash)/Wakala/ajenti wa pesa za,"Refused to Answer/Kataa Kujibu(DO NOT READ OUT)""","Sacco Branch/Sacco agent/ Head Office/""Branch ya sacco/Agent wa sacco/Ofisikuu"""
gender,,,,,,,,,
Female,31,188,174,125,1,4,11748,1,54
Male,35,161,140,43,1,5,8111,0,49



VARIABLE: distance_to_nearest_provider

Data type: str

Number of unique values: 10

Missing values: 0

First 20 values:


count                 20871
unique                   10
top       Under 1 Kilometer
freq                  13684
Name: distance_to_nearest_provider, dtype: object

... (2 outputs left out)


VARIABLE: cost_to_nearest_provider

Data type: str

Number of unique values: 12

Missing values: 0

First 20 values:


count                                                 20871
unique                                                   12
top       Close enough to walk to and fro - No need to s...
freq                                                  17727
Name: cost_to_nearest_provider, dtype: object


Distribution by gender:


cost_to_nearest_provider,#NULL!,"Between KSh 101 - 200/Kati ya KSh 101 -200""","Between KSh 201 - 500/Kati ya KSh 201 - 500""","Between KSh 51 - 100/Kati yaKsh51-100""","Close enough to walk to and fro - No need to spend/""Nikaribu pa kutosha kwenda n","Dont know sijui/(DO NOT READ OUT)""",Far but can walk,"Less than KSh 50/""Chini ya KSh 50""","More than KSh 500/Zaidi ya KSh 500""","Other (SPECIFY)/Nyingine (Taja)""","Refused to Answer Kataa kujibu/(DO NOT READ OUT)""",no financial provider around
gender,,,,,,,,,,,,
Female,20,356,205,539,10407,155,3,490,143,4,1,3
Male,25,202,136,386,7320,51,0,332,92,1,0,0


In [21]:
phone_cols = [
    "owns_phone",
    "phone_internet_capable",
    "phone_access_type",
    "internet_use_frequency",
    "have_ever_accessed_mobile_money",
    "primary_mobile_ownership"
]

available_phone_cols = [
    col for col in phone_cols
    if col in df.columns
]

print("PHONE AND INTERNET AUDIT")
print("-" * 50)

print("Variables found:")
print(available_phone_cols)

for col in available_phone_cols:

    print("\n" + "=" * 50)
    print("VARIABLE:", col)

    print("\nValue counts:")
    display(df[col].value_counts(dropna=False).head(30))

    print("\nMissing values:")
    print(df[col].isnull().sum())

    print("\nDistribution by gender:")

    if gender_col is not None:

        display(
            pd.crosstab(
                df[gender_col],
                df[col],
                normalize="index",
                dropna=False
            ).mul(100).round(2)
        )

PHONE AND INTERNET AUDIT
--------------------------------------------------
Variables found:
['owns_phone', 'phone_internet_capable', 'phone_access_type', 'internet_use_frequency']

VARIABLE: owns_phone

Value counts:


owns_phone
Yes    17350
No      3521
Name: count, dtype: int64


Missing values:
0

Distribution by gender:


owns_phone,No,Yes
gender,,
Female,16.99,83.01
Male,16.70,83.30



VARIABLE: phone_internet_capable

Value counts:


phone_internet_capable
0         10017
1          9069
#NULL!     1785
Name: count, dtype: int64

... (6 outputs left out)


VARIABLE: internet_use_frequency

Value counts:


internet_use_frequency
Never/Situmii"                    9656
Daily/"Kila siku"                 6054
Less ofteN/Mara kwa mara"         2581
Weekly/Kila wiki"                 2111
Monthly/Mara moja kwa mwezi"       404
#NULL!                              44
Refused to Answer/Kataa kujibu      21
Name: count, dtype: int64


Missing values:
0

Distribution by gender:


internet_use_frequency,#NULL!,"Daily/""Kila siku""","Less ofteN/Mara kwa mara""","Monthly/Mara moja kwa mwezi""","Never/Situmii""",Refused to Answer/Kataa kujibu,"Weekly/Kila wiki"""
gender,,,,,,,
Female,0.16,25.25,12.06,1.96,50.80,0.13,9.65
Male,0.28,34.43,12.81,1.90,39.73,0.06,10.79


In [23]:
print("PHONE OWNERSHIP AND CAPABILITY CHECK")
print("-" * 50)

if (
    "owns_phone" in df.columns and
    "phone_internet_capable" in df.columns
):

    print("Phone ownership by internet capability:")

    display(
        pd.crosstab(
            df["owns_phone"],
            df["phone_internet_capable"],
            margins=True,
            dropna=False
        )
    )

    print("\nMissing phone capability by phone ownership:")

    display(
        df.groupby("owns_phone", dropna=False)[
            "phone_internet_capable"
        ]
        .apply(
            lambda x: pd.Series({
                "Total observations": len(x),
                "Missing observations": x.isnull().sum(),
                "Missing percentage": x.isnull().mean() * 100
            })
        )
    )

else:
    print("One or both phone variables were not found.")

PHONE OWNERSHIP AND CAPABILITY CHECK
--------------------------------------------------
Phone ownership by internet capability:


phone_internet_capable,#NULL!,0,1,All
owns_phone,,,,
No,1783,1394,344,3521
Yes,2,8623,8725,17350
All,1785,10017,9069,20871



Missing phone capability by phone ownership:


owns_phone                      
No          Total observations       3521.0
            Missing observations        0.0
            Missing percentage          0.0
Yes         Total observations      17350.0
            Missing observations        0.0
            Missing percentage          0.0
Name: phone_internet_capable, dtype: float64

In [24]:
id_col = "has_any_id"

print("IDENTIFICATION VARIABLE AUDIT")
print("-" * 50)

if id_col in df.columns:

    print("\nID variable value counts:")
    display(df[id_col].value_counts(dropna=False))

    print("\nID variable data type:")
    print(df[id_col].dtype)

    print("\nDistribution by gender:")

    display(
        pd.crosstab(
            df[gender_col],
            df[id_col],
            normalize="index",
            dropna=False
        ).mul(100).round(2)
    )

    print("\nMissing values:")
    print(df[id_col].isnull().sum())

else:
    print("has_any_id was not found.")

IDENTIFICATION VARIABLE AUDIT
--------------------------------------------------

ID variable value counts:


has_any_id
Has ID (National Identity Card (ID),Passport,Alien ID)    18722
No ID                                                      2149
Name: count, dtype: int64


ID variable data type:
str

Distribution by gender:


has_any_id,"Has ID (National Identity Card (ID),Passport,Alien ID)",No ID
gender,,
Female,90.50,9.50
Male,88.55,11.45



Missing values:
0


In [25]:
print("POTENTIAL TARGET LEAKAGE AUDIT")
print("-" * 50)

# Search for columns whose names may indicate overlap
# with financial account ownership or financial-service usage.

keywords = [
    "bank",
    "account",
    "formal",
    "financial",
    "usage",
    "access",
    "mobile_money",
    "provider",
    "barrier",
    "credit",
    "loan",
    "saving"
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    if any(keyword in col_lower for keyword in keywords):

        suspicious_columns.append(col)

print("Variables requiring conceptual review:")
print(suspicious_columns)


# Inspect their relationship with the target

for col in suspicious_columns:

    if col == target:
        continue

    print("\n" + "=" * 60)
    print("VARIABLE:", col)

    print("Number of unique values:", df[col].nunique(dropna=False))

    if df[col].nunique(dropna=False) <= 20:

        print("\nDistribution:")
        display(df[col].value_counts(dropna=False).head(20))

        print("\nRelationship with target:")
        display(
            pd.crosstab(
                df[col],
                df[target],
                normalize="index",
                dropna=False
            ).mul(100).round(2)
        )

    else:

        if pd.api.types.is_numeric_dtype(df[col]):

            print("\nCorrelation with target:")

            display(
                df[[col, target]]
                .corr(numeric_only=True)
                .round(3)
            )

POTENTIAL TARGET LEAKAGE AUDIT
--------------------------------------------------
Variables requiring conceptual review:
['phone_access_type', 'nearest_provider_type', 'cost_to_nearest_provider', 'distance_to_nearest_provider', 'mobile_money_access', 'target_formal_access', 'target_bank_account', 'financial_literacy_score']

VARIABLE: phone_access_type
Number of unique values: 6

Distribution:


phone_access_type
I own a mobile that only I use/Nina simu ambayo ni mimi tu ninaitumia               16888
I don’t own a mobile phone but am able to use someone else’s phone regularly         1738
I don’t own or use a mobile phone/Sina wala situmii simu ya mkononi                  1735
I own a mobile phone that both me and others use/Nina simu ya mkononi ambayo mim      462
#NULL!                                                                                 44
Refused to Answer (DO NOT READ OUT)/Kataa kujibu                                        4
Name: count, dtype: int64


Relationship with target:


target_bank_account,Non-usage,Usage
phone_access_type,,
#NULL!,75.00,25.00
I don’t own a mobile phone but am able to use someone else’s phone regularly,94.42,5.58
I don’t own or use a mobile phone/Sina wala situmii simu ya mkononi,92.39,7.61
I own a mobile phone that both me and others use/Nina simu ya mkononi ambayo mim,83.12,16.88
I own a mobile that only I use/Nina simu ambayo ni mimi tu ninaitumia,71.10,28.90
Refused to Answer (DO NOT READ OUT)/Kataa kujibu,75.00,25.00



VARIABLE: nearest_provider_type
Number of unique values: 9

Distribution:


nearest_provider_type
Mobile Money Agent (for depositing or withdrawing cash)/Wakala/ajenti wa pesa za    19859
Bank Agent/ Post Bank Agent/wakala/ajenti wa benki/ wakala/ajenti waPostbank          349
Commercial Bank/Microfinance Bank/ Post Bank Branch/ Head Office/ ATM/ Benki/taw      314
Don't know/Sijui.(DO NOT READ OUT)"                                                   168
Sacco Branch/Sacco agent/ Head Office/"Branch ya sacco/Agent wa sacco/Ofisikuu"       103
#NULL!                                                                                 66
Microfinance Institution/"Taasisi ndogo la kifedha"                                     9
Insurance company/ agent/"Kampuni la bima/wakalaau ajenti"                              2
Refused to Answer/Kataa Kujibu(DO NOT READ OUT)"                                        1
Name: count, dtype: int64

... (18 outputs left out)


VARIABLE: financial_literacy_score
Number of unique values: 4

Distribution:


financial_literacy_score
Two correct     8580
All correct     8085
One correct     3493
None correct     713
Name: count, dtype: int64


Relationship with target:


target_bank_account,Non-usage,Usage
financial_literacy_score,,
All correct,69.09,30.91
None correct,84.57,15.43
One correct,82.94,17.06
Two correct,76.76,23.24


In [26]:
print("SURVEY DESIGN VARIABLE AUDIT")
print("-" * 50)

survey_candidates = [
    "indWeight",
    "A07",
    "cluster",
    "cluster_number",
    "survey_weight",
    "weight"
]

available_survey_cols = [
    col for col in survey_candidates
    if col in df.columns
]

print("Survey design variables found:")
print(available_survey_cols)

for col in available_survey_cols:

    print("\n" + "=" * 50)
    print("VARIABLE:", col)

    print("\nData type:", df[col].dtype)

    print("\nMissing values:", df[col].isnull().sum())

    print("\nDescriptive statistics:")
    display(df[col].describe())

SURVEY DESIGN VARIABLE AUDIT
--------------------------------------------------
Survey design variables found:
['survey_weight']

VARIABLE: survey_weight

Data type: int64

Missing values: 0

Descriptive statistics:


count    20871.000000
mean      1463.902880
std       1740.019694
min         20.000000
25%        553.000000
50%        994.000000
75%       1773.000000
max      73513.000000
Name: survey_weight, dtype: float64

In [27]:
print("CONSOLIDATED FEATURE QUALITY AUDIT")
print("-" * 60)

audit_cols = [
    "education_level",
    "income_last_month",
    "income_highest_month",
    "income_lowest_month",
    "income_volatility",
    "num_income_sources",
    "wealth_quintile",
    "financial_literacy_score",
    "fl_score",
    "nearest_provider_type",
    "phone_internet_capable",
    "has_any_id"
]

available_audit_cols = [
    col for col in audit_cols
    if col in df.columns
]

audit_summary = pd.DataFrame({
    "Variable": available_audit_cols,
    "Data Type": [
        str(df[col].dtype)
        for col in available_audit_cols
    ],
    "Unique Values": [
        df[col].nunique(dropna=False)
        for col in available_audit_cols
    ],
    "Missing Count": [
        df[col].isnull().sum()
        for col in available_audit_cols
    ],
    "Missing Percentage": [
        df[col].isnull().mean() * 100
        for col in available_audit_cols
    ]
})

display(audit_summary.round(2))

CONSOLIDATED FEATURE QUALITY AUDIT
------------------------------------------------------------


,Variable,Data Type,Unique Values,Missing Count,Missing Percentage
0,education_level,str,5,3078,14.75
1,income_last_month,str,237,0,0.00
2,income_highest_month,str,288,0,0.00
3,income_lowest_month,str,197,0,0.00
4,num_income_sources,int64,7,0,0.00
5,wealth_quintile,str,6,0,0.00
6,financial_literacy_score,str,4,0,0.00
7,nearest_provider_type,str,9,0,0.00
8,phone_internet_capable,str,3,0,0.00
9,has_any_id,str,2,0,0.00


In [28]:
OUTPUT_PATH = PROJECT_DIR / "feature_quality_audit.xlsx"

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:

    # General dataset information
    df.describe(include="all").T.to_excel(
        writer,
        sheet_name="Dataset Summary"
    )

    # Missing values
    missing.to_excel(
        writer,
        sheet_name="Missing Values"
    )

    # Consolidated audit
    audit_summary.to_excel(
        writer,
        sheet_name="Feature Audit",
        index=False
    )

    # Education categories
    if education_col in df.columns:
        df[education_col].value_counts(
            dropna=False
        ).to_excel(
            writer,
            sheet_name="Education"
        )

    # Income variables
    if available_income_cols:
        df[available_income_cols].describe().T.to_excel(
            writer,
            sheet_name="Income"
        )

    # Wealth
    if wealth_col in df.columns:
        df[wealth_col].value_counts(
            dropna=False
        ).to_excel(
            writer,
            sheet_name="Wealth"
        )

print("Audit results saved to:", OUTPUT_PATH)

Audit results saved to: c:\Users\maken\Documents\Strathmore Graduate Studies\DSA 8401 - Applied Machine Learning in Data Science\Assignments\Capstone Project\finaccess-fairness\notebooks\feature_quality_audit.xlsx
